#  Lexos Term Frequency Processors

This notebook provides an in-depth tutorial on the `processors.py` module from the Lexos codebase. These functions are designed to handle a variety of input formats—ranging from raw text to structured document-term matrices (DTMs)—and convert them into **term-frequency dictionaries**.

These dictionaries are foundational for many downstream tasks in Lexos, including:

-  Word cloud generation (`plotly_wordcloud`)
-  Multi-document comparisons (`multicloud`)
-  Any visualization or modeling that depends on token counts

---

##  Why This Module Matters

In a text analysis pipeline, one common challenge is:  
➡️ *“How do I turn different formats of text input into something that can be counted and visualized?”*

The functions in this module answer that challenge by:
- Normalizing different document representations
- Supporting optional document selection (`docs` by index or label)
- Returning simple but consistent Python dictionaries: `{ "word": count }`

---

##  What This Notebook Covers

| Section | Function | Purpose |
|--------|----------|---------|
| 1 | `process_item()` | Handles single items (Doc, Span, list[str], list[Token]) |
| 2 | `process_docs()` | Processes a list of spaCy Docs or Spans |
| 3 | `process_list()` | Handles nested lists (multiple docs) |
| 4 | `filter_docs()` | Filters docs from DTM or DataFrame |
| 5 | `process_dataframe()` | Sums counts from selected columns |
| 6 | `process_dtm()` | Converts DTM → DataFrame → counts |
| 7 | `multicloud_processor()` | Outputs list of term-count dicts |
| 8 | `get_rows()` | Utility to chunk document lists (layout helper) |

---

> Let's start by importing the required libraries and exploring each processor with real examples.


In [35]:
# Imports

# Built-in
from collections import Counter
from pathlib import Path
from itertools import chain

# External libraries
import pandas as pd
import spacy
from spacy.tokens import Doc, Span, Token

# Lexos modules
from lexos.visualization import processors
from lexos.exceptions import LexosException
from lexos.dtm import DTM
from lexos.util import ensure_list


In [ ]:

from lexos.visualization import processors

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load Jane Austen text
text = Path("docs/Austen_Pride.txt").read_text(encoding="utf-8")

# Create spaCy Doc
doc = nlp(text)


## `process_item()` – Process a single document or token list

This function handles one `Doc`, `Span`, or a flat list of strings or `Token` objects. It returns a term-frequency dictionary.


In [37]:
# From full spaCy Doc
output_item = processors.process_item(doc)
print(list(output_item.items())[:10])  # Preview top 10


[(' ', 1), ('Pride', 3), ('and', 3426), ('Prejudice', 1), ('\n', 2262), ('by', 623), ('Jane', 292), ('Austen', 1), ('Chapter', 61), ('1', 1)]



###  Accepted Input Types:
- `list[str]`: e.g., `["word", "cloud", "word"]`
- `list[Token]`: spaCy tokens
- `Doc` or `Span`: full document or partial slice from a spaCy pipeline

###  Output:
A `dict[str, int]` where:
- **keys** = terms
- **values** = counts

This is ideal for single-document use cases, or when you already have a pre-tokenized string list or spaCy object.


In [ ]:
from pathlib import Path
import spacy
from lexos.visualization import processors

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")


file_path = Path("docs/Austen_Pride.txt")

# Read the file
if not file_path.exists():
    raise FileNotFoundError(f"The file {file_path} does not exist. Double-check the path.")
text = file_path.read_text(encoding="utf-8")

# Convert text into spaCy Doc
doc = nlp(text)

# Process full Doc
doc_output = processors.process_item(doc)
print("Full Doc output:", doc_output)

# Process a Span from the doc (e.g., sentence 2)
span = list(doc.sents)[1] if len(list(doc.sents)) > 1 else doc[:10]
span_output = processors.process_item(span)
print("Span output:", span_output)

# Process tokens (as list of strings)
tokens = [token.text for token in doc if not token.is_punct and not token.is_space][:15]
tokens_output = processors.process_item(tokens)
print("List[str] tokens output:", tokens_output)


Full Doc output: {' ': 1, 'Pride': 3, 'and': 3426, 'Prejudice': 1, '\n': 2262, 'by': 623, 'Jane': 292, 'Austen': 1, 'Chapter': 61, '1': 1, 'It': 245, 'is': 834, 'a': 1906, 'truth': 27, 'universally': 3, 'acknowledged': 20, ',': 9112, 'that': 1522, 'single': 11, 'man': 150, 'in': 1795, 'possession': 9, 'of': 3595, 'good': 187, 'fortune': 39, 'must': 307, 'be': 1234, 'want': 44, 'wife': 47, '.': 5014, 'However': 6, 'little': 187, 'known': 58, 'the': 4057, 'feelings': 86, 'or': 297, 'views': 11, 'such': 373, 'may': 186, 'on': 681, 'his': 1191, 'first': 143, 'entering': 9, 'neighbourhood': 28, 'this': 381, 'so': 573, 'well': 188, 'fixed': 21, 'minds': 4, 'surrounding': 2, 'families': 7, 'he': 1100, 'considered': 23, 'rightful': 1, 'property': 8, 'some': 206, 'one': 259, 'other': 209, 'their': 409, 'daughters': 49, '"': 3498, 'My': 112, 'dear': 142, 'Mr.': 786, 'Bennet': 322, 'said': 401, 'lady': 55, 'to': 4108, 'him': 764, 'day': 142, 'have': 831, 'you': 1147, 'heard': 86, 'Netherfield': 7

### Example: Using a Real Text File

In this example, we demonstrate how to use a real `.txt` file from your local `docs/` folder with the `processors` module.

We load the full file, convert it into a spaCy `Doc`, and then pass it through different processor functions:

- `process_item(doc)`: processes the full `Doc`.
- `process_item(span)`: processes a specific `Span` (subsection) of the Doc.
- `process_item(tokens)`: processes a list of raw tokens extracted from the document.

This helps us understand how `Lexos` interprets and transforms different input types into term frequency dictionaries.


##  `process_docs()`: Process a List of Docs or Spans

The `process_docs()` function is designed to handle a **list of spaCy Docs or Spans**, aggregating the tokens across them into a single term-frequency dictionary.

###  Accepted Input:
- `list[Doc]`
- `list[Span]`

###  Optional:
You can pass a `docs` parameter to select specific indices from the list.

###  Output:
A single `dict[str, int]` containing term counts across all selected documents.

---

This is useful when you're working with:
- A small batch of documents you’ve already parsed with spaCy
- Specific document slices you want to include in one visualization


In [39]:
# Split long text into first 3 sentences
docs = list(doc.sents)[:3]
output_docs = processors.process_docs(docs, docs=[0, 1])
print(output_docs)


{' ': 1, 'Pride': 1, 'and': 1, 'Prejudice': 1, '\n': 5, 'by': 1, 'Jane': 1, 'Austen': 1, 'Chapter': 1, '1': 1, 'It': 1, 'is': 3, 'a': 6, 'truth': 2, 'universally': 1, 'acknowledged': 1, ',': 4, 'that': 2, 'single': 1, 'man': 2, 'in': 3, 'possession': 1, 'of': 6, 'good': 1, 'fortune': 1, 'must': 1, 'be': 2, 'want': 1, 'wife': 1, '.': 2, 'However': 1, 'little': 1, 'known': 1, 'the': 4, 'feelings': 1, 'or': 2, 'views': 1, 'such': 1, 'may': 1, 'on': 1, 'his': 1, 'first': 1, 'entering': 1, 'neighbourhood': 1, 'this': 1, 'so': 1, 'well': 1, 'fixed': 1, 'minds': 1, 'surrounding': 1, 'families': 1, 'he': 1, 'considered': 1, 'rightful': 1, 'property': 1, 'some': 1, 'one': 1, 'other': 1, 'their': 1, 'daughters': 1}


In [ ]:
# Example: process_docs with real spaCy Docs from multiple files

from pathlib import Path

# Load all .txt files from your real docs folder
docs_path = Path("docs")
files = sorted(docs_path.glob("*.txt"))

# Load text content and convert each to a spaCy Doc
texts = [f.read_text(encoding="utf-8") for f in files]
docs = [nlp(text) for text in texts]

# Process all documents
all_docs_output = processors.process_docs(docs, docs=None)
print("All docs output:", all_docs_output)

# Process only the first and third documents (index 0 and 2)
subset_output = processors.process_docs(docs, docs=[0, 2])
print("Subset (docs 0 & 2):", subset_output)


All docs output: {' ': 2, 'Pride': 6, 'and': 7118, 'Prejudice': 2, '\n': 13263, 'by': 1448, 'Jane': 340, 'Austen': 3, 'Chapter': 71, '1': 4, 'It': 455, 'is': 1710, 'a': 4261, 'truth': 56, 'universally': 8, 'acknowledged': 35, ',': 20119, 'that': 2984, 'single': 22, 'man': 291, 'in': 3927, 'possession': 22, 'of': 7568, 'good': 381, 'fortune': 86, 'must': 630, 'be': 2674, 'want': 93, 'wife': 102, '.': 9738, 'However': 18, 'little': 365, 'known': 122, 'the': 8395, 'feelings': 162, 'or': 680, 'views': 19, 'such': 761, 'may': 379, 'on': 1417, 'his': 2273, 'first': 315, 'entering': 18, 'neighbourhood': 46, 'this': 803, 'so': 1261, 'well': 401, 'fixed': 48, 'minds': 7, 'surrounding': 7, 'families': 12, 'he': 2128, 'considered': 53, 'rightful': 2, 'property': 20, 'some': 438, 'one': 605, 'other': 409, 'their': 928, 'daughters': 90, '"': 7241, 'My': 181, 'dear': 274, 'Mr.': 1091, 'Bennet': 393, 'said': 864, 'lady': 109, 'to': 8649, 'him': 1458, 'day': 308, 'have': 1720, 'you': 2368, 'heard': 17

### Example: Processing Multiple Documents with `process_docs`

This example demonstrates how to use the `process_docs` function with multiple real `.txt` files from your local `docs` directory.

- The `docs` folder is scanned for `.txt` files.
- Each file is loaded and passed through the `spaCy` NLP pipeline to create a list of `Doc` objects.
- The full list of documents is then processed to generate term frequency counts.
- A subset of documents (by index) can also be processed if needed.

#### Key Function Used
- `processors.process_docs(docs, docs=None)` — returns a dictionary of terms with their frequency across all provided spaCy Docs.

This is useful when working with a corpus of real documents and you want quick insight into which terms appear most frequently.


## This shows:

How to aggregate term frequencies from multiple docs

How to subset specific documents by index


##  `process_list()`: Process Nested Lists of Documents

The `process_list()` function handles **lists of lists**, where each inner list represents a document.  
It supports input types like:

###  Accepted Input:
- `list[list[str]]` — tokenized strings per document
- `list[list[Token]]` — spaCy tokens
- `list[list[Doc]]` or `list[list[Span]]` — spaCy components

###  Optional:
You can use the `docs` parameter to select which inner documents to include (by index).

###  Output:
A single `dict[str, int]` where all terms across the selected documents are counted.

---


In [41]:
# Tokenize into lists of strings per sentence
token_lists = [[token.text for token in sent] for sent in doc.sents][:3]
output_list = processors.process_list(token_lists, docs=[0, 1])
print(output_list)


{' ': 1, 'Pride': 1, 'and': 1, 'Prejudice': 1, '\n': 5, 'by': 1, 'Jane': 1, 'Austen': 1, 'Chapter': 1, '1': 1, 'It': 1, 'is': 3, 'a': 6, 'truth': 2, 'universally': 1, 'acknowledged': 1, ',': 4, 'that': 2, 'single': 1, 'man': 2, 'in': 3, 'possession': 1, 'of': 6, 'good': 1, 'fortune': 1, 'must': 1, 'be': 2, 'want': 1, 'wife': 1, '.': 2, 'However': 1, 'little': 1, 'known': 1, 'the': 4, 'feelings': 1, 'or': 2, 'views': 1, 'such': 1, 'may': 1, 'on': 1, 'his': 1, 'first': 1, 'entering': 1, 'neighbourhood': 1, 'this': 1, 'so': 1, 'well': 1, 'fixed': 1, 'minds': 1, 'surrounding': 1, 'families': 1, 'he': 1, 'considered': 1, 'rightful': 1, 'property': 1, 'some': 1, 'one': 1, 'other': 1, 'their': 1, 'daughters': 1}


In [ ]:
from pathlib import Path
from lexos.visualization import processors

# Step 1: Load all .txt files from the docs folder
docs_folder = Path("docs")
txt_files = sorted(docs_folder.glob("*.txt"))

# Step 2: Print loaded filenames
print("Loaded files:")
for i, file in enumerate(txt_files):
    print(f"{i}: {file.name}")

# Step 3: Read and tokenize each file (basic whitespace tokenizer)
tokenized_docs = [file.read_text(encoding="utf-8").lower().split() for file in txt_files]

# Step 4: Process all documents
all_docs_output = processors.process_list(tokenized_docs, docs=None)
print("\nAll docs:", all_docs_output)

# Step 5: Process a subset (e.g., documents 0 and 2)
subset_output = processors.process_list(tokenized_docs, docs=[0, 2])
print("Subset (docs 0 & 2):", subset_output)


Loaded files:
0: Austen_Pride.txt
1: Austen_Pride_sm.txt
2: Austen_Sense.txt

All docs: {'pride': 33, 'and': 6976, 'prejudice': 8, 'by': 1445, 'jane': 177, 'austen': 3, 'chapter': 121, '1': 4, 'it': 2541, 'is': 1614, 'a': 4280, 'truth': 32, 'universally': 7, 'acknowledged,': 13, 'that': 2846, 'single': 16, 'man': 171, 'in': 3965, 'possession': 18, 'of': 7506, 'good': 304, 'fortune,': 31, 'must': 622, 'be': 2567, 'want': 87, 'wife.': 12, 'however': 57, 'little': 324, 'known': 95, 'the': 8893, 'feelings': 111, 'or': 661, 'views': 16, 'such': 782, 'may': 369, 'on': 1398, 'his': 2401, 'first': 282, 'entering': 17, 'neighbourhood,': 18, 'this': 781, 'so': 1180, 'well': 255, 'fixed': 44, 'minds': 5, 'surrounding': 7, 'families,': 3, 'he': 2417, 'considered': 45, 'rightful': 2, 'property': 12, 'some': 440, 'one': 545, 'other': 305, 'their': 996, 'daughters.': 17, '"my': 75, 'dear': 189, 'mr.': 1057, 'bennet,"': 13, 'said': 760, 'lady': 302, 'to': 8623, 'him': 982, 'day,': 65, '"have': 12, 'yo

### Example: Processing Multiple Real `.txt` Files with `process_list`

In this example, we demonstrate how to use the `process_list` function from the `lexos.visualization.processors` module with actual `.txt` files stored in the `docs/` folder of your project.


1. **Loads `.txt` files from the `docs/` folder**:
   - Uses Python's `Path` library to read all `.txt` files in the directory.
   - Sorts them alphabetically so the order is predictable and repeatable.

2. **Displays file names**:
   - For reference, the names of all loaded files are printed with their indices. This helps users know which file corresponds to which index during selection.

3. **Tokenizes each file**:
   - Each file’s contents are converted to lowercase and split by whitespace to form a list of words (tokens). This is a very basic form of tokenization.

4. **Processes all documents**:
   - Passes the full list of tokenized documents to `process_list`, which returns a combined term-frequency dictionary for all tokens across all documents.

5. **Processes a subset of documents**:
   - Demonstrates how to use the `docs` parameter to focus on just specific documents (in this case, documents with indices 0 to 2).

> This approach is useful for quick, lightweight processing of text without needing NLP libraries like spaCy, while still supporting multiple document inputs.


##  `filter_docs()`: Select Columns from a DTM or DataFrame

The `filter_docs()` function is a utility that filters specific documents (columns) from a **pandas DataFrame**, usually representing a DTM.

It supports two ways to filter:

###  Accepted `docs` values:
- `list[int]` → select by column index
- `list[str]` → select by column label

If no `docs` are provided, the original DataFrame is returned unchanged.

---

###  Typical Use Case:
This function is used internally by:
- `process_dataframe()`
- `process_dtm()`
- `multicloud_processor()` (when input is a DataFrame)

It allows flexible control over which documents to analyze in larger corpora.


In [ ]:
from pathlib import Path
import pandas as pd
from lexos.visualization import processors

# Step 1: Load all text files from the docs folder
docs_folder = Path("docs")
txt_files = sorted(docs_folder.glob("*.txt"))

# Step 2: Create a simple term-document matrix manually (mocked from real data)
# Note: You might build this with CountVectorizer, but here we'll do it manually
terms = ["data", "science", "lexos"]
data = {}

for i, file in enumerate(txt_files):
    text = file.read_text(encoding="utf-8").lower()
    counts = [text.count(term) for term in terms]
    data[file.stem] = counts  # use filename (without extension) as label

# Create the DataFrame
df = pd.DataFrame(data, index=terms)

# Step 3: Demonstrate filtering

# 1. Filter by index
filtered_by_index = processors.filter_docs(df, docs=[0, 2])
print("Filtered by index:\n", filtered_by_index)

# 2. Filter by filename/label
file_labels = list(data.keys())
if len(file_labels) >= 2:
    filtered_by_label = processors.filter_docs(df, docs=[file_labels[1]])
    print("\nFiltered by label:\n", filtered_by_label)

# 3. No filtering
no_filtering = processors.filter_docs(df)
print("\nNo filtering:\n", no_filtering)


Filtered by index:
          Austen_Pride  Austen_Sense
data                0             0
science             6            14
lexos               0             0

Filtered by label:
          Austen_Pride_sm
data                   0
science                1
lexos                  0

No filtering:
          Austen_Pride  Austen_Pride_sm  Austen_Sense
data                0                0             0
science             6                1            14
lexos               0                0             0


### Example: Using `filter_docs` with `.txt` Files

In this example, we demonstrate how to use the `filter_docs` function to selectively include certain documents from a term-document matrix built from real text files.

#### What this code does:

1. **Loads all `.txt` files** from the `docs/` folder and reads their contents.

2. **Manually creates a term-document matrix**:
   - We count how many times specific terms appear in each file.
   - We store those counts in a pandas `DataFrame`, with terms as rows and file names (without extension) as columns.

3. **Applies different filters** using `filter_docs()`:
   - **By index**: Selects documents based on their order in the `DataFrame` (e.g., first and third columns).
   - **By label**: Filters using the actual file names (as column labels).
   - **Without filtering**: Returns the full DataFrame when no `docs` parameter is passed.

> This is a lightweight way to demonstrate document selection logic without needing full NLP pipelines or vectorizers.


##  `process_dataframe()`: Convert a DataFrame into a Term Frequency Dictionary

The `process_dataframe()` function takes a **term-document matrix** in the form of a `pandas.DataFrame` and:

1. Optionally filters it to select specific documents (columns)
2. Sums the counts for each term across the selected docs
3. Returns a dictionary: `{term: total_count}`

---

###  Expected Input:
- A `DataFrame` where:
  - **Rows** = terms
  - **Columns** = document labels or indices
  - **Values** = term frequencies per document

###  Optional:
- `docs`: specify a list of columns to include using **labels** or **indices**

---

### Output:
A `dict[str, int]` — total frequency per term across selected documents.

This is commonly used with `.to_df()` output from a Lexos `DTM`.


In [ ]:
from pathlib import Path
import pandas as pd
from lexos.visualization import processors

# Step 1: Load real term-document matrix from .txt files
docs_folder = Path("docs")
txt_files = sorted(docs_folder.glob("*.txt"))
doc_names = [file.stem for file in txt_files]

# Step 2: Read and tokenize files (basic)
tokenized_docs = [file.read_text(encoding="utf-8").lower().split() for file in txt_files]

# Step 3: Use process_list to get the term frequency DTM
dtm_dict_list = [processors.process_item(doc) for doc in tokenized_docs]
dtm = pd.DataFrame(dtm_dict_list).T  # transpose so terms are rows
dtm.columns = doc_names


# 1. Process all documents
all_docs_output = processors.process_dataframe(dtm)
print("All docs processed:\n", all_docs_output)

# 2. Process selected documents by name
subset_output = processors.process_dataframe(dtm, docs=[doc_names[1], doc_names[2]])
print("\nSubset (second & third docs):\n", subset_output)

# 3. Process selected documents by index
index_subset_output = processors.process_dataframe(dtm, docs=[0, 2])
print("\nSubset (index 0 & 2):\n", index_subset_output)


All docs processed:
 {'pride': 33.0, 'and': 6976.0, 'prejudice': 8.0, 'by': 1445.0, 'jane': 177.0, 'austen': 3.0, 'chapter': 121.0, '1': 4.0, 'it': 2541.0, 'is': 1614.0, 'a': 4280.0, 'truth': 32.0, 'universally': 7.0, 'acknowledged,': 13.0, 'that': 2846.0, 'single': 16.0, 'man': 171.0, 'in': 3965.0, 'possession': 18.0, 'of': 7506.0, 'good': 304.0, 'fortune,': 31.0, 'must': 622.0, 'be': 2567.0, 'want': 87.0, 'wife.': 12.0, 'however': 57.0, 'little': 324.0, 'known': 95.0, 'the': 8893.0, 'feelings': 111.0, 'or': 661.0, 'views': 16.0, 'such': 782.0, 'may': 369.0, 'on': 1398.0, 'his': 2401.0, 'first': 282.0, 'entering': 17.0, 'neighbourhood,': 18.0, 'this': 781.0, 'so': 1180.0, 'well': 255.0, 'fixed': 44.0, 'minds': 5.0, 'surrounding': 7.0, 'families,': 3.0, 'he': 2417.0, 'considered': 45.0, 'rightful': 2.0, 'property': 12.0, 'some': 440.0, 'one': 545.0, 'other': 305.0, 'their': 996.0, 'daughters.': 17.0, '"my': 75.0, 'dear': 189.0, 'mr.': 1057.0, 'bennet,"': 13.0, 'said': 760.0, 'lady': 30

### Example: `process_dataframe` with documents

This example demonstrates how to generate and analyze a term-document matrix from multiple real `.txt` files using `process_dataframe`.

**Workflow Overview:**

1. **Load `.txt` files** from your `docs/` folder using `pathlib.Path.glob()`.
2. **Tokenize** the contents of each file using a basic `str.split()` tokenizer.
3. **Convert each document** into a term-frequency dictionary using `process_item()`.
4. **Build a term-document matrix** (`pandas.DataFrame`) where:
   - **Rows** represent terms
   - **Columns** represent documents
5. Use `process_dataframe()` to compute aggregated term frequencies:
   - Across all documents
   - Across a subset of documents (by name or index)

This is helpful when you want to work directly with raw `.txt` files and still leverage the full power of Lexos's processor utilities.


##  `process_dtm()`: Process a Lexos DTM to Term Frequencies

The `process_dtm()` function takes a **Lexos `DTM` object** and converts it into a term-frequency dictionary.

This is a convenience wrapper around `process_dataframe()`:
- It first calls `.to_df()` on the `DTM`
- Then performs the same filtering and aggregation

---

###  Parameters:
- `dtm`: A Lexos `DTM` instance
- `docs` *(optional)*: Document labels or indices to select

###  Output:
A `dict[str, int]` — term frequencies summed across the selected documents.

---

###  Use Case:
Use this when working directly with Lexos's `DTM` pipeline, instead of manually converting to `DataFrame`.

It is most useful when integrating into larger workflows using the Lexos DTM engine.


In [ ]:
from pathlib import Path
import spacy
from lexos.visualization import processors

# Load your real .txt files from the docs folder
docs_path = Path("docs")
text_files = sorted(docs_path.glob("*.txt"))
texts = [file.read_text(encoding="utf-8") for file in text_files]

# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Example 1: Use flat list of tokens from first two files
flat_list = texts[0].lower().split() + texts[1].lower().split()
output_flat = processors.process_item(flat_list)
print("List[str] output:", output_flat)

# Example 2: Use spaCy Doc from third file (if it exists)
if len(texts) > 2:
    doc = nlp(texts[2])
    output_doc = processors.process_item(doc)
    print("Doc output:", output_doc)

    # Example 3: Use a Span from that Doc
    if len(doc) >= 5:
        span = doc[2:5]
        output_span = processors.process_item(span)
        print("Span output:", output_span)
    else:
        print("Span could not be created. Doc too short.")
else:
    print("Not enough documents to create third Doc.")


List[str] output: {'pride': 30, 'and': 3720, 'prejudice': 6, 'by': 711, 'jane': 176, 'austen': 2, 'chapter': 71, '1': 2, 'it': 1269, 'is': 939, 'a': 2250, 'truth': 18, 'universally': 4, 'acknowledged,': 8, 'that': 1610, 'single': 10, 'man': 103, 'in': 2068, 'possession': 10, 'of': 3967, 'good': 181, 'fortune,': 20, 'must': 343, 'be': 1335, 'want': 48, 'wife.': 7, 'however': 17, 'little': 188, 'known': 51, 'the': 4822, 'feelings': 62, 'or': 320, 'views': 12, 'such': 436, 'may': 202, 'on': 735, 'his': 1399, 'first': 141, 'entering': 11, 'neighbourhood,': 14, 'this': 419, 'so': 601, 'well': 134, 'fixed': 24, 'minds': 4, 'surrounding': 3, 'families,': 3, 'he': 1420, 'considered': 23, 'rightful': 2, 'property': 7, 'some': 231, 'one': 283, 'other': 176, 'their': 500, 'daughters.': 13, '"my': 55, 'dear': 119, 'mr.': 889, 'bennet,"': 13, 'said': 418, 'lady': 180, 'to': 4574, 'him': 550, 'day,': 29, '"have': 8, 'you': 1250, 'heard': 86, 'netherfield': 52, 'park': 10, 'let': 72, 'at': 881, 'last


---

###  `processors.process_item()` – Output Explanation

The `process_item()` function in Lexos takes different types of textual input (like a list of words, a spaCy Doc, or a Span) and returns a dictionary of term frequencies — showing how often each word or token appears.

#### Example 1: `List[str]` Input

When you pass a plain list of lowercase strings (e.g., split words from `.lower().split()`), the function returns a simple frequency count for each word:

This is ideal when working with raw text and you don’t need special NLP parsing.

---

####  Example 2: `Doc` (spaCy object) Input

When you pass a **spaCy Doc** (like `nlp(text)`), Lexos tokenizes the text using NLP rules and includes case sensitivity, punctuation, and even special characters like newlines (`'\n'`).

**Output:**

```python
{'SENSE': 1, 'AND': 1, 'SENSIBILITY': 1, '\n': 10598, '(': 30, ')': 29, ...}
```

Useful when working with structured or rich text from books, articles, etc.

---

#### Example 3: `Span` Input

A **Span** is a slice of a `Doc` (e.g., `doc[2:5]`), which selects a phrase or sentence fragment. The output gives frequencies for just that small selection.

**Output:**

```python
{'SENSIBILITY': 1, '\n': 1, 'by': 1}
```

Great for analyzing a specific part of the document.

---

### Summary Table

| Input Type  | Description                    | Output Content                         |
| ----------- | ------------------------------ | -------------------------------------- |
| `List[str]` | Simple list of lowercase words | Lowercase word counts                  |
| `Doc`       | Full NLP-processed document    | Case-sensitive tokens, includes punct. |
| `Span`      | Subset of tokens from a Doc    | Token counts from the selected span    |

This flexibility makes `process_item()` powerful for various levels of text analysis — from raw word lists to complex NLP pipelines.

---



### Using `process_list()` with Multiple Documents

If you have a list of tokenized documents (e.g., `List[List[str]]`), `process_list()` lets you compute term frequencies:

- `docs=None` will process **all** documents.
- You can also pass indices (e.g., `[0, 2]`) to filter specific documents.

This function also works with:
- Lists of spaCy Docs
- Lists of Spans
- Lists of spaCy Tokens

The output is a single frequency dictionary across the selected documents.


In [ ]:
from pathlib import Path
import spacy
from lexos.visualization import processors

# Load the language model
nlp = spacy.load("en_core_web_sm")

# Load your real text files
text1 = Path("docs/Austen_Pride_sm.txt").read_text(encoding="utf-8")
text2 = Path("docs/Austen_Sense.txt").read_text(encoding="utf-8")
text3 = Path("docs/Austen_Pride_sm.txt").read_text(encoding="utf-8")

# Convert to spaCy Docs
docs = [nlp(text) for text in [text1, text2, text3]]

# Step 1: Process ALL documents
output_all = processors.process_docs(docs, docs=None)
print("All Docs Output:\n", output_all)

# Step 2: Process SELECTED documents only
output_selected = processors.process_docs(docs, docs=[0, 2])
print("\nDocs [0, 2] Output:\n", output_selected)


All Docs Output:
 {' ': 2, 'Pride': 6, 'and': 4109, 'Prejudice': 2, '\n': 11404, 'by': 913, 'Jane': 95, 'Austen': 3, 'Chapter': 20, '1': 4, 'It': 234, 'is': 1029, 'a': 2674, 'truth': 32, 'universally': 6, 'acknowledged': 18, ',': 12197, 'that': 1638, 'single': 14, 'man': 165, 'in': 2365, 'possession': 15, 'of': 4385, 'good': 224, 'fortune': 57, 'must': 367, 'be': 1588, 'want': 56, 'wife': 63, '.': 5443, 'However': 14, 'little': 200, 'known': 68, 'the': 4821, 'feelings': 80, 'or': 415, 'views': 10, 'such': 438, 'may': 218, 'on': 799, 'his': 1223, 'first': 185, 'entering': 11, 'neighbourhood': 26, 'this': 475, 'so': 766, 'well': 238, 'fixed': 31, 'minds': 5, 'surrounding': 6, 'families': 7, 'he': 1173, 'considered': 33, 'rightful': 2, 'property': 15, 'some': 258, 'one': 389, 'other': 218, 'their': 575, 'daughters': 55, '"': 4515, 'My': 82, 'dear': 158, 'Mr.': 432, 'Bennet': 142, 'said': 542, 'lady': 65, 'to': 5027, 'him': 765, 'day': 184, 'have': 973, 'you': 1421, 'heard': 98, 'Netherfie

### Using `process_docs()` for spaCy Documents

The `process_docs()` function takes a list of `spaCy` Docs or Spans and converts them into a frequency dictionary.

- It extracts tokens from the input documents.
- You can process **all documents** or **filter by index** (e.g., `[1, 2]`).
- Works well for already-parsed content using spaCy’s NLP pipeline.


In [ ]:
import spacy
from pathlib import Path
from lexos.visualization import processors

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Step 1: Load all .txt files from the docs folder
docs_path = Path("docs")
txt_files = sorted(docs_path.glob("*.txt"))

# Step 2: Read each file and convert to spaCy Docs
texts = [file.read_text(encoding="utf-8") for file in txt_files]
docs = [nlp(text) for text in texts]

# Step 3: Process all spaCy Docs
output_multi_all = processors.multicloud_processor(docs)
print("Multi all (Docs):", output_multi_all)

# Step 4: Process only selected Docs by index (e.g., 0 and 2)
output_multi_filtered = processors.multicloud_processor(docs, docs=[0, 2])
print("Multi filtered (Docs) [0,2]:", output_multi_filtered)


Multi all (Docs): [{' ': 1, 'Pride': 3, 'and': 3426, 'Prejudice': 1, '\n': 2262, 'by': 623, 'Jane': 292, 'Austen': 1, 'Chapter': 61, '1': 1, 'It': 245, 'is': 834, 'a': 1906, 'truth': 27, 'universally': 3, 'acknowledged': 20, ',': 9112, 'that': 1522, 'single': 11, 'man': 150, 'in': 1795, 'possession': 9, 'of': 3595, 'good': 187, 'fortune': 39, 'must': 307, 'be': 1234, 'want': 44, 'wife': 47, '.': 5014, 'However': 6, 'little': 187, 'known': 58, 'the': 4057, 'feelings': 86, 'or': 297, 'views': 11, 'such': 373, 'may': 186, 'on': 681, 'his': 1191, 'first': 143, 'entering': 9, 'neighbourhood': 28, 'this': 381, 'so': 573, 'well': 188, 'fixed': 21, 'minds': 4, 'surrounding': 2, 'families': 7, 'he': 1100, 'considered': 23, 'rightful': 1, 'property': 8, 'some': 206, 'one': 259, 'other': 209, 'their': 409, 'daughters': 49, '"': 3498, 'My': 112, 'dear': 142, 'Mr.': 786, 'Bennet': 322, 'said': 401, 'lady': 55, 'to': 4108, 'him': 764, 'day': 142, 'have': 831, 'you': 1147, 'heard': 86, 'Netherfield':

### Using `multicloud_processor()` for Multi-Document Input

The `multicloud_processor()` is used when you're dealing with multiple documents and want to generate **multiple word clouds**.

- Each input should be a list of term lists (or other supported formats like Docs or dicts).
- The function returns a list of frequency dictionaries — one per document.
- You can filter by index using the `docs` argument.

Great for multi-panel visualizations like grid word clouds or comparison dashboards.


In [ ]:
from pathlib import Path
from lexos.visualization import processors

# Step 1: Get all filenames from the docs folder
docs_path = Path("docs")
txt_files = sorted(docs_path.glob("*.txt"))

# Step 2: Use filenames (without extension) as labels
labels = [file.stem for file in txt_files]

# Step 3: Split labels into rows of 3 for visualization layout
rows = list(processors.get_rows(labels, n=3))

# Step 4: Print each row
for i, row in enumerate(rows):
    print(f"Row {i+1}: {row}")


Row 1: ['Austen_Pride', 'Austen_Pride_sm', 'Austen_Sense']


### Organizing Word Clouds into Grid Rows with `get_rows()`

The `get_rows()` function is a simple utility that splits a list of documents (or any items) into evenly spaced rows.

This is especially useful when creating grid-style word cloud visualizations (e.g., 3x3).

- `lst`: Your list of document labels or word cloud data.
- `n`: Number of items per row (e.g., `n=3` for 3-column rows).

The output is a generator that yields rows one at a time.
